In [ ]:
!pip install openmeteo-requests requests-cache retry-requests psycopg2-binary sqlalchemy -q

import openmeteo_requests
import requests_cache
import requests
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from retry_requests import retry
from datetime import datetime
from google.colab import userdata

print("✓ All libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 120.4 MB/s eta 0:00:00
✓ All libraries ready


In [ ]:
# Cell 2 — Database setup

SUPABASE_URI = userdata.get('SUPABASE_URI')
engine = create_engine(SUPABASE_URI)

def init_db():
    with engine.connect() as conn:
        conn.execute(text("""

            CREATE TABLE IF NOT EXISTS weather_hourly (
                timestamp           TIMESTAMP PRIMARY KEY,
                temperature_f       REAL,
                apparent_temp_f     REAL,
                precipitation_in    REAL,
                rain_in             REAL,
                snowfall_in         REAL,
                snow_depth_in       REAL,
                wind_speed_mph      REAL
            );

            CREATE TABLE IF NOT EXISTS subway_hourly (
                timestamp           TIMESTAMP,
                borough             TEXT,
                subway_ridership    BIGINT,
                PRIMARY KEY (timestamp, borough)
            );

            CREATE TABLE IF NOT EXISTS bus_hourly (
                timestamp           TIMESTAMP,
                borough             TEXT,
                bus_ridership       BIGINT,
                PRIMARY KEY (timestamp, borough)
            );

            CREATE TABLE IF NOT EXISTS bridges_tunnels_hourly (
                timestamp               TIMESTAMP,
                borough                 TEXT,
                bridge_tunnel_volume    BIGINT,
                PRIMARY KEY (timestamp, borough)
            );

            CREATE TABLE IF NOT EXISTS traffic_volume_hourly (
                timestamp               TIMESTAMP,
                borough                 TEXT,
                street_traffic_volume   BIGINT,
                sensor_count            INTEGER,
                PRIMARY KEY (timestamp, borough)
            );

            CREATE TABLE IF NOT EXISTS master_hourly (
                timestamp               TIMESTAMP,
                borough                 TEXT,
                temperature_f           REAL,
                apparent_temp_f         REAL,
                precipitation_in        REAL,
                rain_in                 REAL,
                snowfall_in             REAL,
                snow_depth_in           REAL,
                wind_speed_mph          REAL,
                subway_ridership        BIGINT,
                bus_ridership           BIGINT,
                bridge_tunnel_volume    BIGINT,
                PRIMARY KEY (timestamp, borough)
            );

        """))
        conn.commit()
    print("✓ Connected to Supabase")
    print("  Tables created:")
    print("    - weather_hourly")
    print("    - subway_hourly")
    print("    - bus_hourly")
    print("    - bridges_tunnels_hourly")
    print("    - traffic_volume_hourly")
    print("    - master_hourly")

init_db()

✓ Connected to Supabase
  Tables created:
    - weather_hourly
    - subway_hourly
    - bus_hourly
    - bridges_tunnels_hourly
    - traffic_volume_hourly
    - master_hourly


In [ ]:
# Cell 3 — Open-Meteo weather fetch

def fetch_weather():
    print("Fetching weather data from Open-Meteo...")

    # Set up cached session with retry
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    params = {
        "latitude": 40.7143,
        "longitude": -74.006,
        "start_date": "2020-01-01",
        "end_date": "2024-12-31",
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "precipitation",
            "rain",
            "snowfall",
            "snow_depth",
            "wind_speed_10m"
        ],
        "timezone": "America/New_York",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch"
    }

    responses = openmeteo.weather_api(
        "https://historical-forecast-api.open-meteo.com/v1/forecast",
        params=params
    )
    response = responses[0]

    # Parse hourly data
    hourly = response.Hourly()
    timestamps = pd.date_range(
        start=pd.Timestamp(hourly.Time(), unit="s", tz="America/New_York"),
        end=pd.Timestamp(hourly.TimeEnd(), unit="s", tz="America/New_York"),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    df = pd.DataFrame({
        "timestamp":          timestamps,
        "temperature_f":      hourly.Variables(0).ValuesAsNumpy(),
        "apparent_temp_f":    hourly.Variables(1).ValuesAsNumpy(),
        "precipitation_in":   hourly.Variables(2).ValuesAsNumpy(),
        "rain_in":            hourly.Variables(3).ValuesAsNumpy(),
        "snowfall_in":        hourly.Variables(4).ValuesAsNumpy(),
        "snow_depth_in":      hourly.Variables(5).ValuesAsNumpy(),
        "wind_speed_mph":     hourly.Variables(6).ValuesAsNumpy(),
    })

    # Normalize timestamp — strip timezone, floor to hour
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None).dt.floor("h")

    print(f"  Rows fetched:  {len(df):,}")
    print(f"  Date range:    {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Nulls:         {df.isnull().sum().sum()}")
    return df


def write_weather(df):
    print("Writing weather data to Supabase...")
    df.to_sql(
        "weather_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to weather_hourly")


# --- Run ---
weather_df = fetch_weather()
print(weather_df.head())
write_weather(weather_df)

Fetching weather data from Open-Meteo...
  Rows fetched:  43,848
  Date range:    2019-12-31 23:00:00 → 2024-12-31 22:00:00
  Nulls:         0
            timestamp  temperature_f  apparent_temp_f  precipitation_in  \
0 2019-12-31 23:00:00      40.473499        31.766161               0.0   
1 2020-01-01 00:00:00      39.663502        30.068617               0.0   
2 2020-01-01 01:00:00      38.133499        28.694639               0.0   
3 2020-01-01 02:00:00      37.413498        28.589113               0.0   
4 2020-01-01 03:00:00      37.143501        27.480818               0.0   

   rain_in  snowfall_in  snow_depth_in  wind_speed_mph  
0      0.0          0.0            0.0       11.364753  
1      0.0          0.0            0.0       12.945641  
2      0.0          0.0            0.0       12.207545  
3      0.0          0.0            0.0       10.530545  
4      0.0          0.0            0.0       12.432971  
Writing weather data to Supabase...
  ✓ 43,848 rows written to w

In [ ]:
# Cell 4 — MTA Subway Hourly Ridership fetch

def fetch_subway_ridership():
    print("Fetching MTA Subway Hourly Ridership 2020-2024...")

    url = "https://data.ny.gov/resource/wujg-7c2s.json"
    all_dfs = []

    for year in range(2020, 2025):
        print(f"\n  Year {year}...")
        offset = 0
        limit = 50000

        while True:
            params = {
                "$select": "transit_timestamp, borough, sum(ridership) AS subway_ridership",
                "$where":  f"transit_mode='subway' AND transit_timestamp >= '{year}-01-01T00:00:00' AND transit_timestamp <= '{year}-12-31T23:00:00'",
                "$group":  "transit_timestamp, borough",
                "$order":  "transit_timestamp ASC",
                "$limit":  limit,
                "$offset": offset
            }

            try:
                r = requests.get(url, params=params, timeout=120)
                r.raise_for_status()
                batch = pd.DataFrame(r.json())

                if batch.empty:
                    print(f"    No more data at offset {offset:,} — done")
                    break

                all_dfs.append(batch)
                offset += limit
                print(f"    Fetched {offset:,} rows so far...")

            except Exception as e:
                print(f"    ✗ Error at offset {offset:,}: {e}")
                break

    if not all_dfs:
        raise ValueError("No subway data returned.")

    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n  Total rows fetched: {len(df):,}")

    # --- Normalize ---
    df["subway_ridership"] = pd.to_numeric(df["subway_ridership"], errors="coerce")
    df["timestamp"] = pd.to_datetime(df["transit_timestamp"]).dt.floor("h")
    df = df.drop(columns=["transit_timestamp"])

    # Standardize borough names to match across all datasets
    borough_map = {
        "Manhattan":     "Manhattan",
        "Brooklyn":      "Brooklyn",
        "Queens":        "Queens",
        "Bronx":         "Bronx",
        "Staten Island": "Staten Island"
    }
    df["borough"] = df["borough"].map(borough_map)
    df = df[df["borough"].notna()]

    # Aggregate to timestamp + borough
    df = (
        df.groupby(["timestamp", "borough"])
        .agg(subway_ridership=("subway_ridership", "sum"))
        .reset_index()
    )

    print(f"  Rows after aggregation: {len(df):,}")
    print(f"  Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Boroughs: {sorted(df['borough'].unique())}")
    print(f"  Nulls: {df.isnull().sum().sum()}")
    return df


def write_subway(df):
    print("\nWriting subway data to Supabase...")
    df.to_sql(
        "subway_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to subway_hourly")


# --- Run ---
subway_df = fetch_subway_ridership()
print(subway_df.head(10))
write_subway(subway_df)

Fetching MTA Subway Hourly Ridership 2020-2024...

  Year 2020...
    Fetched 50,000 rows so far...
    No more data at offset 50,000 — done

  Year 2021...
    Fetched 50,000 rows so far...
    No more data at offset 50,000 — done

  Year 2022...
    Fetched 50,000 rows so far...
    No more data at offset 50,000 — done

  Year 2023...
    Fetched 50,000 rows so far...
    No more data at offset 50,000 — done

  Year 2024...
    Fetched 50,000 rows so far...
    No more data at offset 50,000 — done

  Total rows fetched: 174,415
  Rows after aggregation: 174,415
  Date range: 2020-01-01 00:00:00 → 2024-12-31 23:00:00
  Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
  Nulls: 0
            timestamp    borough  subway_ridership
0 2020-01-01 00:00:00      Bronx            3054.0
1 2020-01-01 00:00:00   Brooklyn           25021.0
2 2020-01-01 00:00:00  Manhattan           78314.0
3 2020-01-01 00:00:00     Queens            5299.0
4 2020-01-01 01:00:00      Bronx  

In [ ]:
# Cell 5 — MTA Bus Hourly Ridership fetch (borough by borough)

BOROUGH_ROUTE_FILTERS = {
    "Brooklyn":      "(starts_with(bus_route, 'BM') OR starts_with(bus_route, 'B')) AND NOT starts_with(bus_route, 'BX')",
    "Bronx":         "(starts_with(bus_route, 'BXM') OR starts_with(bus_route, 'BX'))",
    "Manhattan":     "starts_with(bus_route, 'M')",
    "Queens":        "(starts_with(bus_route, 'QM') OR starts_with(bus_route, 'Q'))",
    "Staten Island": "(starts_with(bus_route, 'SIM') OR starts_with(bus_route, 'S') OR starts_with(bus_route, 'X'))",
}

def fetch_bus_ridership():
    print("Fetching MTA Bus Hourly Ridership 2020-2024...")

    url = "https://data.ny.gov/resource/kv7t-n8in.json"
    all_dfs = []

    for borough, route_filter in BOROUGH_ROUTE_FILTERS.items():
        for year in range(2020, 2025):
            print(f"  {borough} {year}...", end=" ")
            offset = 0
            limit = 50000
            year_dfs = []

            while True:
                params = {
                    "$select": f"transit_timestamp, sum(ridership) AS bus_ridership",
                    "$where":  f"transit_timestamp >= '{year}-01-01T00:00:00' "
                               f"AND transit_timestamp <= '{year}-12-31T23:00:00' "
                               f"AND ({route_filter})",
                    "$group":  "transit_timestamp",
                    "$order":  "transit_timestamp ASC",
                    "$limit":  limit,
                    "$offset": offset
                }

                try:
                    r = requests.get(url, params=params, timeout=120)
                    r.raise_for_status()
                    batch = pd.DataFrame(r.json())

                    if batch.empty:
                        break

                    year_dfs.append(batch)
                    offset += limit

                except Exception as e:
                    print(f"\n    ✗ Error at offset {offset:,}: {e}")
                    break

            if year_dfs:
                df_chunk = pd.concat(year_dfs, ignore_index=True)
                df_chunk["borough"] = borough
                all_dfs.append(df_chunk)
                print(f"{len(df_chunk):,} rows ✓")
            else:
                print("no data")

    if not all_dfs:
        raise ValueError("No bus data returned.")

    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n  Total rows fetched: {len(df):,}")

    # --- Normalize ---
    df["bus_ridership"] = pd.to_numeric(df["bus_ridership"], errors="coerce")
    df["timestamp"] = pd.to_datetime(df["transit_timestamp"]).dt.floor("h")
    df = df.drop(columns=["transit_timestamp"])

    # Aggregate to timestamp + borough in case of any overlap
    df = (
        df.groupby(["timestamp", "borough"])
        .agg(bus_ridership=("bus_ridership", "sum"))
        .reset_index()
    )

    print(f"  Rows after aggregation: {len(df):,}")
    print(f"  Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Boroughs: {sorted(df['borough'].unique())}")
    print(f"  Nulls: {df.isnull().sum().sum()}")
    return df


def write_bus(df):
    print("\nWriting bus data to Supabase...")
    df.to_sql(
        "bus_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to bus_hourly")


# --- Run ---
bus_df = fetch_bus_ridership()
print(bus_df.head(10))
write_bus(bus_df)

Fetching MTA Bus Hourly Ridership 2020-2024...
  Brooklyn 2020... 8,784 rows ✓
  Brooklyn 2021... 8,760 rows ✓
  Brooklyn 2022... 8,760 rows ✓
  Brooklyn 2023... 8,760 rows ✓
  Brooklyn 2024... 8,784 rows ✓
  Bronx 2020... 8,784 rows ✓
  Bronx 2021... 8,760 rows ✓
  Bronx 2022... 8,760 rows ✓
  Bronx 2023... 8,760 rows ✓
  Bronx 2024... 8,784 rows ✓
  Manhattan 2020... 8,784 rows ✓
  Manhattan 2021... 8,760 rows ✓
  Manhattan 2022... 8,760 rows ✓
  Manhattan 2023... 8,760 rows ✓
  Manhattan 2024... 8,784 rows ✓
  Queens 2020... 8,784 rows ✓
  Queens 2021... 8,760 rows ✓
  Queens 2022... 8,760 rows ✓
  Queens 2023... 8,760 rows ✓
  Queens 2024... 8,784 rows ✓
  Staten Island 2020... 8,784 rows ✓
  Staten Island 2021... 8,760 rows ✓
  Staten Island 2022... 8,760 rows ✓
  Staten Island 2023... 8,760 rows ✓
  Staten Island 2024... 
    ✗ Error at offset 50,000: HTTPSConnectionPool(host='data.ny.gov', port=443): Read timed out. (read timeout=120)
8,784 rows ✓

  Total rows fetched: 219,240


In [ ]:
# Cell 6 — MTA Bridges & Tunnels Hourly Traffic fetch

FACILITY_BOROUGH_MAP = {
    "21": "Bronx",        # Robert F. Kennedy Bridge Bronx
    "22": "Manhattan",    # Robert F. Kennedy Bridge Manhattan
    "23": "Bronx",        # Bronx - Whitestone Bridge
    "24": "Manhattan",    # Henry Hudson Bridge
    "25": "Queens",       # Marine Parkway Bridge
    "26": "Queens",       # Cross Bay Bridge
    "27": "Manhattan",    # Queens Midtown Tunnel
    "28": "Manhattan",    # Hugh L. Carey Tunnel
    "29": "Bronx",        # Throgs Neck Bridge
    "30": "Brooklyn",     # Verrazzano - Narrows Bridge
}

def fetch_bridges_tunnels():
    print("Fetching MTA Bridges & Tunnels Hourly Traffic 2020-2024...")

    url = "https://data.ny.gov/resource/ebfx-2m7v.json"
    all_dfs = []

    for year in range(2020, 2025):
        print(f"\n  Year {year}...")
        offset = 0
        limit = 50000

        while True:
            params = {
                "$select": "transit_timestamp, facility_id, sum(traffic_count) AS bridge_tunnel_volume",
                "$where":  f"transit_timestamp >= '{year}-01-01T00:00:00' "
                           f"AND transit_timestamp <= '{year}-12-31T23:00:00'",
                "$group":  "transit_timestamp, facility_id",
                "$order":  "transit_timestamp ASC",
                "$limit":  limit,
                "$offset": offset
            }

            try:
                r = requests.get(url, params=params, timeout=120)
                r.raise_for_status()
                batch = pd.DataFrame(r.json())

                if batch.empty:
                    print(f"    No more data at offset {offset:,} — done")
                    break

                all_dfs.append(batch)
                offset += limit
                print(f"    Fetched {offset:,} rows so far...")

            except Exception as e:
                print(f"    ✗ Error at offset {offset:,}: {e}")
                break

    if not all_dfs:
        raise ValueError("No bridges & tunnels data returned.")

    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n  Total rows fetched: {len(df):,}")

    # --- Normalize ---
    df["bridge_tunnel_volume"] = pd.to_numeric(df["bridge_tunnel_volume"], errors="coerce")
    df["timestamp"] = pd.to_datetime(df["transit_timestamp"]).dt.floor("h")
    df = df.drop(columns=["transit_timestamp"])

    # Map facility_id to borough
    df["borough"] = df["facility_id"].map(FACILITY_BOROUGH_MAP)

    # Report unmapped facilities
    unmapped = df[df["borough"].isna()]["facility_id"].unique()
    if len(unmapped) > 0:
        print(f"  ⚠ Unmapped facility IDs dropped: {unmapped}")
    df = df[df["borough"].notna()]
    df = df.drop(columns=["facility_id"])

    # Aggregate to timestamp + borough
    df = (
        df.groupby(["timestamp", "borough"])
        .agg(bridge_tunnel_volume=("bridge_tunnel_volume", "sum"))
        .reset_index()
    )

    print(f"  Rows after aggregation: {len(df):,}")
    print(f"  Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Boroughs: {sorted(df['borough'].unique())}")
    print(f"  Nulls: {df.isnull().sum().sum()}")
    return df


def write_bridges_tunnels(df):
    print("\nWriting bridges & tunnels data to Supabase...")
    df.to_sql(
        "bridges_tunnels_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to bridges_tunnels_hourly")


# --- Run ---
bt_df = fetch_bridges_tunnels()
print(bt_df.head(10))
write_bridges_tunnels(bt_df)

Fetching MTA Bridges & Tunnels Hourly Traffic 2020-2024...

  Year 2020...
    Fetched 50,000 rows so far...
    Fetched 100,000 rows so far...
    No more data at offset 100,000 — done

  Year 2021...
    Fetched 50,000 rows so far...
    Fetched 100,000 rows so far...
    No more data at offset 100,000 — done

  Year 2022...
    Fetched 50,000 rows so far...
    Fetched 100,000 rows so far...
    No more data at offset 100,000 — done

  Year 2023...
    Fetched 50,000 rows so far...
    Fetched 100,000 rows so far...
    No more data at offset 100,000 — done

  Year 2024...
    Fetched 50,000 rows so far...
    Fetched 100,000 rows so far...
    No more data at offset 100,000 — done

  Total rows fetched: 438,403
  Rows after aggregation: 175,349
  Date range: 2020-01-01 00:00:00 → 2024-12-31 23:00:00
  Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']
  Nulls: 0
            timestamp    borough  bridge_tunnel_volume
0 2020-01-01 00:00:00      Bronx                  8097
1 2020-

In [ ]:
# Cell 7 — NYC Automated Traffic Volume Counts fetch


def fetch_traffic_volume():
    print("Fetching NYC Automated Traffic Volume Counts 2020-2024...")

    url = "https://data.cityofnewyork.us/resource/7ym2-wayt.json"
    all_dfs = []

    for year in range(2020, 2025):
        print(f"  Year {year}...", end=" ")

        params = {
            "$select": "yr, m, d, hh, boro, "
                       "sum(vol) AS street_traffic_volume, "
                       "count(segmentid) AS sensor_count",
            "$where":  f"yr='{year}'",
            "$group":  "yr, m, d, hh, boro",
            "$order":  "yr ASC, m ASC, d ASC, hh ASC",
            "$limit":  50000
        }

        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            batch = pd.DataFrame(r.json())

            if batch.empty:
                print("no data")
                continue

            all_dfs.append(batch)
            print(f"{len(batch):,} rows ✓")

        except Exception as e:
            print(f"✗ Error: {e}")

    if not all_dfs:
        raise ValueError("No traffic volume data returned.")

    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n  Total rows fetched: {len(df):,}")

    # --- Normalize ---
    df["street_traffic_volume"] = pd.to_numeric(df["street_traffic_volume"], errors="coerce")
    df["sensor_count"]          = pd.to_numeric(df["sensor_count"],          errors="coerce")
    df["yr"] = pd.to_numeric(df["yr"], errors="coerce")
    df["m"]  = pd.to_numeric(df["m"],  errors="coerce")
    df["d"]  = pd.to_numeric(df["d"],  errors="coerce")
    df["hh"] = pd.to_numeric(df["hh"], errors="coerce")

    # Build timestamp from separate yr, m, d, hh columns
    df["timestamp"] = pd.to_datetime(dict(
        year=df["yr"], month=df["m"], day=df["d"], hour=df["hh"]
    ), errors="coerce")
    df["timestamp"] = df["timestamp"].dt.floor("h")
    df = df.drop(columns=["yr", "m", "d", "hh"])

    # Standardize borough name to match other datasets
    borough_map = {
        "Manhattan":     "Manhattan",
        "Brooklyn":      "Brooklyn",
        "Queens":        "Queens",
        "Bronx":         "Bronx",
        "Staten Island": "Staten Island"
    }
    df["borough"] = df["boro"].map(borough_map)
    df = df[df["borough"].notna()]
    df = df.drop(columns=["boro"])

    # Drop rows where timestamp couldn't be parsed
    df = df[df["timestamp"].notna()]

    print(f"  Rows after normalization: {len(df):,}")
    print(f"  Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Boroughs: {sorted(df['borough'].unique())}")
    print(f"  Nulls: {df.isnull().sum().sum()}")
    print(f"  Avg sensors per reading: {df['sensor_count'].mean():.1f}")
    return df


def write_traffic_volume(df):
    print("\nWriting traffic volume data to Supabase...")
    df.to_sql(
        "traffic_volume_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to traffic_volume_hourly")


# --- Run ---
traffic_df = fetch_traffic_volume()
print(traffic_df.head(10))
write_traffic_volume(traffic_df)

Fetching NYC Automated Traffic Volume Counts 2020-2024...
  Year 2020... 4,728 rows ✓
  Year 2021... 9,405 rows ✓
  Year 2022... 7,525 rows ✓
  Year 2023... 7,931 rows ✓
  Year 2024... 9,414 rows ✓

  Total rows fetched: 39,003
  Rows after normalization: 39,003
  Date range: 2020-01-20 00:00:00 → 2024-12-11 23:00:00
  Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
  Nulls: 0
  Avg sensors per reading: 9.9
   street_traffic_volume  sensor_count           timestamp borough
0                      9             8 2020-01-20 00:00:00  Queens
1                      4             8 2020-01-20 01:00:00  Queens
2                      6             8 2020-01-20 02:00:00  Queens
3                      4             8 2020-01-20 03:00:00  Queens
4                      7             8 2020-01-20 04:00:00  Queens
5                      6             8 2020-01-20 05:00:00  Queens
6                     29             8 2020-01-20 06:00:00  Queens
7                    105     

In [ ]:
# Cell 8 — Build master_hourly table

def build_master():
    print("Building master_hourly table...")

    # --- Load all staged tables from Supabase ---
    print("\n  Loading staged tables...")
    with engine.connect() as conn:
        weather_df = pd.read_sql("SELECT * FROM weather_hourly",          conn)
        subway_df  = pd.read_sql("SELECT * FROM subway_hourly",           conn)
        bus_df     = pd.read_sql("SELECT * FROM bus_hourly",              conn)
        bt_df      = pd.read_sql("SELECT * FROM bridges_tunnels_hourly",  conn)

    print(f"    weather_hourly:           {len(weather_df):>10,} rows")
    print(f"    subway_hourly:            {len(subway_df):>10,} rows")
    print(f"    bus_hourly:               {len(bus_df):>10,} rows")
    print(f"    bridges_tunnels_hourly:   {len(bt_df):>10,} rows")

    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
    subway_df["timestamp"]  = pd.to_datetime(subway_df["timestamp"])
    bus_df["timestamp"]     = pd.to_datetime(bus_df["timestamp"])
    bt_df["timestamp"]      = pd.to_datetime(bt_df["timestamp"])

    # Union all timestamp+borough combinations from the three MTA datasetso
    print("\n  Building timestamp + borough spine...")
    spine = pd.concat([
        subway_df[["timestamp", "borough"]],
        bus_df[["timestamp", "borough"]],
        bt_df[["timestamp", "borough"]]
    ]).drop_duplicates().reset_index(drop=True)
    print(f"    Spine rows: {len(spine):,}")

    # --- Join MTA datasets onto spine ---
    print("\n  Joining MTA datasets...")
    master = (
        spine
        .merge(subway_df,  on=["timestamp", "borough"], how="left")
        .merge(bus_df,     on=["timestamp", "borough"], how="left")
        .merge(bt_df,      on=["timestamp", "borough"], how="left")
    )

    # Weather join on timestamp only
    print("  Broadcasting weather across boroughs...")
    master = master.merge(weather_df, on="timestamp", how="left")

    master = master[
        master["timestamp"].between("2020-01-01", "2024-12-31 23:00:00")
    ]


    master = master[[
        "timestamp",
        "borough",
        "temperature_f",
        "apparent_temp_f",
        "precipitation_in",
        "rain_in",
        "snowfall_in",
        "snow_depth_in",
        "wind_speed_mph",
        "subway_ridership",
        "bus_ridership",
        "bridge_tunnel_volume"
    ]]

    # --- Summary ---
    print(f"\n  Master table shape:  {master.shape}")
    print(f"  Date range:          {master['timestamp'].min()} → {master['timestamp'].max()}")
    print(f"  Boroughs:            {sorted(master['borough'].unique())}")
    print(f"\n  Null counts:")
    for col in master.columns:
        nulls = master[col].isnull().sum()
        if nulls > 0:
            print(f"    {col:<30} {nulls:>8,} nulls  ({nulls/len(master)*100:.1f}%)")

    return master


def write_master(df):
    print("\nWriting master table to Supabase...")
    df.to_sql(
        "master_hourly",
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000
    )
    print(f"  ✓ {len(df):,} rows written to master_hourly")


# --- Run ---
master_df = build_master()
print(master_df.head(10))
write_master(master_df)

Building master_hourly table...

  Loading staged tables...
    weather_hourly:               43,848 rows
    subway_hourly:               174,415 rows
    bus_hourly:                  219,240 rows
    bridges_tunnels_hourly:      175,349 rows

  Building timestamp + borough spine...
    Spine rows: 219,240

  Joining MTA datasets...
  Broadcasting weather across boroughs...

  Master table shape:  (219265, 12)
  Date range:          2020-01-01 00:00:00 → 2024-12-31 23:00:00
  Boroughs:            ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']

  Null counts:
    temperature_f                        30 nulls  (0.0%)
    apparent_temp_f                      30 nulls  (0.0%)
    precipitation_in                     30 nulls  (0.0%)
    rain_in                              30 nulls  (0.0%)
    snowfall_in                          30 nulls  (0.0%)
    snow_depth_in                        30 nulls  (0.0%)
    wind_speed_mph                       30 nulls  (0.0%)
    subway_ri